In [ ]:
import os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import pandas as pd
import numpy as np

from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import Ridge, Lasso, ElasticNet, BayesianRidge, ARDRegression
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import LocalOutlierFactor

import warnings
warnings.filterwarnings('ignore')

print("✓ Imports done")

In [ ]:
df_full = pd.read_csv('../../../data/002_processed_data/xy_df.csv')
df_kaggle = pd.read_csv('../../../data/002_processed_data/x_kaggle_df.csv')
df_full = df_full.dropna(subset=['homo', 'lumo'])

X = df_full.drop(columns=['smiles', 'homo', 'lumo', 'stability']).values
y_homo = df_full['homo'].values
y_lumo = df_full['lumo'].values
X_kaggle = df_kaggle.drop(columns=['smiles']).values

print(f"Train: {X.shape}, Kaggle: {X_kaggle.shape}")

In [ ]:
# HOMO: gmm_5_diag, n=80, k=50
gmm_homo = GaussianMixture(n_components=5, covariance_type='diag', random_state=42)
gmm_homo.fit(X_kaggle)
scores_homo = gmm_homo.score_samples(X)
idx_homo = np.argsort(scores_homo)[-80:]

# LUMO: gmm_5_full, n=90, k=50
gmm_lumo = GaussianMixture(n_components=5, covariance_type='full', random_state=42)
gmm_lumo.fit(X_kaggle)
scores_lumo = gmm_lumo.score_samples(X)
idx_lumo = np.argsort(scores_lumo)[-90:]

In [ ]:
sel_homo = SelectKBest(f_regression, k=50)
X_homo_sel = sel_homo.fit_transform(X[idx_homo], y_homo[idx_homo])

model_homo = ARDRegression()

# Cross-validation
cv_homo = cross_val_score(model_homo, X_homo_sel, y_homo[idx_homo], cv=5, scoring='neg_mean_absolute_error')
print(f"HOMO CV MAE: {-cv_homo.mean():.4f} ± {cv_homo.std():.4f}")

# Fit final model
model_homo.fit(X_homo_sel, y_homo[idx_homo])
pred_homo = model_homo.predict(sel_homo.transform(X_kaggle))
pred_homo = np.clip(pred_homo, y_homo.min(), y_homo.max())

In [ ]:
sel_lumo = SelectKBest(f_regression, k=50)
X_lumo_sel = sel_lumo.fit_transform(X[idx_lumo], y_lumo[idx_lumo])

model_lumo = ARDRegression()

# Cross-validation
cv_lumo = cross_val_score(model_lumo, X_lumo_sel, y_lumo[idx_lumo], cv=5, scoring='neg_mean_absolute_error')
print(f"LUMO CV MAE: {-cv_lumo.mean():.4f} ± {cv_lumo.std():.4f}")

# Fit final model
model_lumo.fit(X_lumo_sel, y_lumo[idx_lumo])
pred_lumo = model_lumo.predict(sel_lumo.transform(X_kaggle))
pred_lumo = np.clip(pred_lumo, y_lumo.min(), y_lumo.max())

In [ ]:
print("=" * 50)
print("FINAL RESULTS")
print("=" * 50)
print(f"\nHOMO CV MAE: {-cv_homo.mean():.4f}")
print(f"LUMO CV MAE: {-cv_lumo.mean():.4f}")
print(f"Average MAE: {(-cv_homo.mean() + -cv_lumo.mean()) / 2:.4f}")
print(f"\nPredictions:")
print(f"  HOMO: {pred_homo.mean():.4f} ± {pred_homo.std():.4f}")
print(f"  LUMO: {pred_lumo.mean():.4f} ± {pred_lumo.std():.4f}")

In [ ]:
submission = pd.DataFrame({
    'smiles': df_kaggle['smiles'],
    'homo': pred_homo,
    'lumo': pred_lumo
})
submission.to_csv('../../../data/003_kaggle_competition_data/25261_regression_sample_submission.csv', index=False)
print(f"✓ Saved")
print(f"HOMO: {pred_homo.mean():.4f} ± {pred_homo.std():.4f}")
print(f"LUMO: {pred_lumo.mean():.4f} ± {pred_lumo.std():.4f}")